# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RamKapoor22/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row = One unique content page (url) per client.

**Time Window:** Mid-panel observation month (month = 2026-03) with a 90-day lookback for aggregated features.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (5):** content_age_days, days_since_last_update, impressions_90d, avg_position, word_count (all knowable at decision time).

**Label:** is_declining_label (1 if trend_direction == 'down', 0 otherwise).

**Context:** url, client_id (identifiers).

**Excluded:** future_clicks_30d (Excluded to prevent target data leakage).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# Clone repo in Colab if not present
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    if os.path.basename(os.getcwd()) != REPO_DIR:
        os.chdir(REPO_DIR)

# Load Dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. Query 1: Verify Grain & Counts
print("--- Query 1: Grain & Row Counts ---")
total_rows = len(df)
print(f"Total rows in slice: {total_rows} (One row = One page)")

# 2. Query 2: Date / Window Span & Missing Values
print("\n--- Query 2: Missing Values & Window Span ---")
print(f"Missing impressions: {df['impressions_90d'].isna().sum()}")
print(f"Max Content Age: {df['content_age_days'].max()} days")

# 3. Query 3: Availability (IS TRUE filter)
print("\n--- Query 3: Availability Check ---")
active_rows = df[df['impressions_90d'].notna() & (df['impressions_90d'] > 0)]
print(f"Active rows surviving IS TRUE filter: {len(active_rows)} / {len(df)}")

# --- Feature Frame & The Trap Experiment ---
features_df = pd.DataFrame({
    'content_age_days': df['content_age_days'],
    'days_since_last_update': df['days_since_last_update'],
    'impressions_90d': df['impressions_90d'],
    'avg_position': df['avg_position'],
    'word_count': df['word_count']
})
features_df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# Deliberate feature leakage test
features_df['leaked_future_metric'] = features_df['is_declining_label'] * 0.99
print(f"\n[LEAK TRAP] Correlation with Leaked Column: {features_df['leaked_future_metric'].corr(features_df['is_declining_label']):.4f}")

# Remove leak
features_df.drop(columns=['leaked_future_metric'], inplace=True)
print("[LEAK FIX] Removed leaked column. Honest feature frame verified.")

--- Query 1: Grain & Row Counts ---
Total rows in slice: 30000 (One row = One page)

--- Query 2: Missing Values & Window Span ---
Missing impressions: 0
Max Content Age: 564 days

--- Query 3: Availability Check ---
Active rows surviving IS TRUE filter: 30000 / 30000

[LEAK TRAP] Correlation with Leaked Column: 1.0000
[LEAK FIX] Removed leaked column. Honest feature frame verified.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation of the Slice:** The 90-day aggregated performance window cannot detect instant Google algorithm updates or sudden short-term seasonal search intent shifts (e.g., micro-trends within a 7-day window).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.